# Lab 4: How Precise Is a Benchmark Score?


## How this lab works

Labs are not graded. Work with a classmate or on your own, whichever helps you more.

1. Run the setup cells below.
2. Follow along as the instructor works through the example.
3. Change the code when the lab asks you to, and write short answers where you see TODO.
4. Save a copy before you leave. Most labs set up that week's homework, and you can reuse the code.


In [ ]:
# Run this cell once. It downloads a small course helper file, then uses it
# to download this week's data and check that every file arrived intact.
from pathlib import Path
import urllib.request

HELPER_URL = "https://raw.githubusercontent.com/skgallagher/stat-methods-ai-public/main/course_helpers/course_setup.py"
if not Path("course_setup.py").exists():
    urllib.request.urlretrieve(HELPER_URL, "course_setup.py")

from course_setup import setup_course
data_dir = setup_course('camera_traps')


In [ ]:
# Libraries and a fixed random seed used in this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2027
np.random.seed(SEED)  # makes random results repeatable


## The situation

Wildlife biologists put motion-triggered cameras along trails to learn which animals live in an area and when they are active. Every time something moves in front of a camera, it takes a quick burst of three photos, called a **sequence**. One camera can take thousands of photos a month, and many are empty: wind moved a branch, or the light changed. Someone has to look through all of them.

A team is considering an AI model, MegaDetector, to sort the photos. For each photo, the model gives a probability that an animal is present. Photos it flags go to a person, and photos it calls empty may never be looked at. That makes one kind of mistake costly. If the model calls a photo empty when an animal is there, that sighting can be lost, and small or rare animals are exactly the ones a survey can least afford to miss.

Before relying on the model, the team asks two questions:

1. How often is the model right on photos like theirs?
2. How sure can we be about that number?

Today we answer both for a small set of cameras whose photos were labeled by people. These labeled photos are the team's **benchmark**, and the **accuracy** is the benchmark score: the share of photos where the model's call (animal or empty) matches the person's label. It is an estimate: a different set of photos would give a somewhat different number. An interval tells the team which accuracies are plausible given the photos we have.

We use the **analysis cameras** (6 cameras) and the **lab's held-out cameras** (3 other cameras). Homework 4 uses a third set of cameras, which we do not touch today.

## 1. Load the photos

Each row of the table is one photo, called a **frame**. For each frame we know which camera took it, which sequence it belongs to, what a person labeled it, and what the model said.

In [ ]:
# Additional libraries for this lab
from scipy.stats import norm
from statsmodels.stats.proportion import proportion_confint

# Read the Camera Traps tables and join them, one row per frame.
camera_dir = data_dir / "camera_traps"
metadata = pd.read_csv(camera_dir / "metadata.csv")
model_outputs = pd.read_csv(camera_dir / "model_outputs.csv")
image_features = pd.read_csv(camera_dir / "image_features.csv")
splits = pd.read_csv(camera_dir / "splits.csv")
frames = pd.merge(metadata, model_outputs, on="image_id")
frames = pd.merge(frames, image_features, on="image_id")
frames = pd.merge(frames, splits, on="image_id")

# correct = 1 if the model's call matches the person's label, 0 if not.
frames["correct"] = (frames["vision_pred"] == frames["animal_present"]).astype(int)

analysis = frames[frames["split"] == "analysis"]
lab_holdout = frames[frames["split"] == "holdout_camera"]

print("Analysis cameras:   ", len(analysis), "frames in", analysis["sequence_id"].nunique(), "sequences")
print("Lab held-out cameras:", len(lab_holdout), "frames in", lab_holdout["sequence_id"].nunique(), "sequences")

## 2. Look at the photos

**Why this step:** before estimating anything, we need to see what the model is looking at, what makes a photo hard, and how the photos relate to each other. That last point will change our interval.

In [ ]:
print("Frames with an animal:   ", (analysis["animal_present"] == 1).sum())
print("Frames without an animal:", (analysis["animal_present"] == 0).sum())

# Does every frame in a sequence have the same label?
labels_per_sequence = analysis.groupby("sequence_id")["animal_present"].nunique()
print("Sequences whose frames disagree on the label:", (labels_per_sequence > 1).sum())

This teaching set was built with equal numbers of photos with and without animals. Real camera-trap collections are usually mostly empty photos, so accuracy here may not match accuracy in the field.

Here are the first frames of three random sequences with an animal and three without. Each title shows the true label and the model's probability that an animal is present.

In [ ]:
def show_frames(rows, title):
    """Show the images for a few rows of the frames table, side by side."""
    fig, axes = plt.subplots(1, len(rows), figsize=(4.3 * len(rows), 3.4))
    for i in range(len(rows)):
        row = rows.iloc[i]
        image = plt.imread(data_dir / row["image_path"])
        axes[i].imshow(image)
        axes[i].set_title(f"{row['species_label']}, P(animal) = {row['vision_prob']:.2f}")
        axes[i].axis("off")
    fig.suptitle(title)
    plt.show()

first_frames = analysis[analysis["frame_index"] == 0]
with_animal = first_frames[first_frames["animal_present"] == 1].sample(3, random_state=SEED)
without_animal = first_frames[first_frames["animal_present"] == 0].sample(3, random_state=SEED)

show_frames(with_animal, "Animal present")
show_frames(without_animal, "No animal")

Which animals are easy to see, and which would be easy to miss? What do the empty frames have in common?

**What you notice:** TODO

Now look at every frame of one sequence with an animal and one without.

In [ ]:
sequence_with = analysis[analysis["sequence_id"] == with_animal["sequence_id"].iloc[0]]
sequence_without = analysis[analysis["sequence_id"] == without_animal["sequence_id"].iloc[0]]

show_frames(sequence_with.sort_values("frame_index"), "One sequence with an animal")
show_frames(sequence_without.sort_values("frame_index"), "One sequence without an animal")

The frames in a sequence are nearly the same picture. We can check this with numbers. Below, each dot pairs the first and second frames of a sequence. On the left, both frames come from the **same** sequence. On the right, we pair each first frame with the second frame of a **different**, randomly chosen sequence.

In [ ]:
first = analysis[analysis["frame_index"] == 0].sort_values("sequence_id")
second = analysis[analysis["frame_index"] == 1].sort_values("sequence_id")
second_shuffled = second.sample(frac=1, random_state=SEED)   # break up the pairs

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row_number, column in enumerate(["brightness", "vision_prob"]):
    x = first[column].to_numpy()
    y_same = second[column].to_numpy()
    y_different = second_shuffled[column].to_numpy()

    axes[row_number, 0].scatter(x, y_same)
    axes[row_number, 0].set_title(f"Same sequence: r = {np.corrcoef(x, y_same)[0, 1]:.2f}")
    axes[row_number, 1].scatter(x, y_different)
    axes[row_number, 1].set_title(f"Different sequences: r = {np.corrcoef(x, y_different)[0, 1]:.2f}")

    for axis in axes[row_number]:
        axis.set_xlabel(f"{column}, first frame")
        axis.set_ylabel(f"{column}, second frame")
plt.tight_layout()
plt.show()

The top row is a property of the images (brightness). The bottom row is the model's output (its probability that an animal is present). Frames from the same sequence are nearly copies of each other, and so are the model's outputs. What does that suggest about whether the model's **errors** are independent across frames?

**Your answer:** TODO

## 3. Instructor run: choosing an interval

**Why this step:** the team's report will include an interval for accuracy. When accuracy is high and there are not many photos, as at a single camera, the familiar textbook (Wald) interval can give answers that make no sense.

Suppose the model is correct on 18 of 20 photos. We compare three intervals:

- **Wald**, the textbook interval, which we compute by hand;
- **Wilson**, which you derive in Homework 4;
- **Clopper–Pearson**, often called the *exact* interval. It is built directly from the binomial distribution instead of a normal approximation. In `statsmodels` it is `method="beta"`.

Then we try 20 of 20.

### Your prediction

TODO: before running the code, write what you expect in 1-2 sentences.


In [ ]:
z = norm.ppf(0.975)   # 1.96

def wald_interval(k, n, z):
    p_hat = k / n
    plus_minus = z * np.sqrt(p_hat * (1 - p_hat) / n)
    return p_hat - plus_minus, p_hat + plus_minus

for k in [18, 20]:
    wald_lower, wald_upper = wald_interval(k, 20, z)
    wilson_lower, wilson_upper = proportion_confint(k, 20, method="wilson")
    exact_lower, exact_upper = proportion_confint(k, 20, method="beta")
    print(k, "of 20")
    print("   Wald:           ", round(wald_lower, 3), "to", round(wald_upper, 3))
    print("   Wilson:         ", round(wilson_lower, 3), "to", round(wilson_upper, 3))
    print("   Clopper-Pearson:", round(exact_lower, 3), "to", round(exact_upper, 3))

What goes wrong with the Wald interval at 18 of 20? At 20 of 20? How do Wilson and Clopper–Pearson differ?

**Your answer:** TODO

## 4. Does the interval keep its promise?

**Why this step:** a 95% interval makes a promise. Across many repeated studies, it should contain the true accuracy about 95% of the time. In real data we never know the true accuracy, but in a simulation we do, so we can check the promise. Here we make up a true accuracy, simulate 2,000 test sets, and count how often each interval covers it.

In [ ]:
rng = np.random.default_rng(SEED)
true_p = 0.95       # the true accuracy we made up
n = 20              # test images in each simulated test set
n_repeats = 2000

wald_covers = 0
wilson_covers = 0
exact_covers = 0
wilson_total_width = 0
exact_total_width = 0
for r in range(n_repeats):
    k = rng.binomial(n, true_p)   # number correct in one simulated test set
    wald_lower, wald_upper = wald_interval(k, n, z)
    wilson_lower, wilson_upper = proportion_confint(k, n, method="wilson")
    exact_lower, exact_upper = proportion_confint(k, n, method="beta")
    if wald_lower <= true_p <= wald_upper:
        wald_covers = wald_covers + 1
    if wilson_lower <= true_p <= wilson_upper:
        wilson_covers = wilson_covers + 1
    if exact_lower <= true_p <= exact_upper:
        exact_covers = exact_covers + 1
    wilson_total_width = wilson_total_width + (wilson_upper - wilson_lower)
    exact_total_width = exact_total_width + (exact_upper - exact_lower)

print("Coverage, Wald:           ", wald_covers / n_repeats)
print("Coverage, Wilson:         ", wilson_covers / n_repeats)
print("Coverage, Clopper-Pearson:", exact_covers / n_repeats)
print("Average width, Wilson:         ", round(wilson_total_width / n_repeats, 3))
print("Average width, Clopper-Pearson:", round(exact_total_width / n_repeats, 3))

Now change `n` to 50 and then 200, and try `true_p = 0.70`. When does Wald do about as well as the others? Compare with Homework 4, Problem 1(a)(iv).

Clopper–Pearson is built to cover the truth **at least** 95% of the time, whatever the true accuracy. Wilson aims for about 95% on average and can dip a little below. If the team needed that guarantee, which would you choose, and what does the guarantee cost?

**What you found:** TODO

## 5. A camera with no mistakes

**Why this step:** some cameras had no mistakes at all. Could the team stop checking photos from those cameras? That depends on how much a run of error-free photos really tells us.

In [ ]:
for camera in sorted(analysis["camera_id"].unique()):
    camera_frames = analysis[analysis["camera_id"] == camera]
    n_correct = camera_frames["correct"].sum()
    n_sequences = camera_frames["sequence_id"].nunique()
    print(camera, ":", n_correct, "of", len(camera_frames), "correct, in", n_sequences, "sequences")

Three cameras have 18 of 18 correct. A one-sided 95% upper bound on the error rate for 18 error-free independent frames is exactly $1 - 0.05^{1/18}$, and the rule of three approximates it by $3/18$. This exact bound is the one-sided Clopper–Pearson interval from section 3, applied to zero errors. Homework 4, Problem 1(b) derives it.

In [ ]:
n_frames = 18
print("Exact bound:  ", round(1 - 0.05 ** (1 / n_frames), 3))
print("Rule of three:", round(3 / n_frames, 3))

Those 18 frames come from only 6 sequences. Using what you saw in section 2, why might they be worth less than 18 independent frames?

**Your answer:** TODO

## 6. Look at the mistakes

**Why this step:** the team worries most about missed animals, so look at every one. A count of errors cannot tell us whether a "mistake" is the model's fault.

In [ ]:
# Frames where a person labeled an animal but the model said empty.
misses = analysis[(analysis["animal_present"] == 1) & (analysis["vision_pred"] == 0)]
print(misses[["sequence_id", "frame_index", "species_label", "vision_prob"]])

# Show every frame of each sequence that contains a miss.
for sequence_id in misses["sequence_id"].unique():
    sequence_frames = analysis[analysis["sequence_id"] == sequence_id]
    show_frames(sequence_frames.sort_values("frame_index"), "A sequence with a miss")

In which frames does the model "miss"? Look closely: can you see the animal in those frames?

**What you see:** TODO

Count the misses by position in the sequence, for both sets of cameras.

In [ ]:
both = pd.concat([analysis, lab_holdout])
both_misses = both[(both["animal_present"] == 1) & (both["vision_pred"] == 0)]
for frame_index in [0, 1, 2]:
    count = (both_misses["frame_index"] == frame_index).sum()
    print("Frame", frame_index + 1, "of 3:", count, "misses")

In section 2 you found that the three frames of a sequence always share one label. The label describes the **whole burst**. When an animal walks out of view, the later frames still say "animal," and a model that correctly calls them empty is scored as wrong.

Why might the labels have been made this way? Think about how much work it is to label three frames instead of one burst, and what the people who built the dataset wanted to learn. Also, when we built this teaching set, we kept only sequences whose three frames share a label. What might that choice have done?

**Your answer:** TODO

The team sends whole bursts to a person, not single frames. At that level, how many bursts with an animal did the model flag in at least one frame?

In [ ]:
animal_sequences = analysis[analysis["animal_present"] == 1]["sequence_id"].unique()
caught = 0
for sequence_id in animal_sequences:
    sequence_frames = analysis[analysis["sequence_id"] == sequence_id]
    if sequence_frames["vision_pred"].max() == 1:
        caught = caught + 1
print("Bursts with an animal, flagged in at least one frame:", caught, "of", len(animal_sequences))

For the rest of the lab we keep frame-level accuracy, so you can compare with Homework 4. Remember what it measures: agreement with **burst** labels, not with what is in each frame.

## 7. An interval that respects the sequences

**Why this step:** section 2 showed that the three photos in a sequence are nearly copies, and section 6 showed that mistakes come in bunches within a sequence, here mostly because of the labels. If we treat 108 photos as 108 independent pieces of evidence, we overstate how much we know. The **sequence bootstrap** builds an interval by resampling whole sequences instead of single photos. This is the function you will reuse in Homework 4.

In [ ]:
def sequence_bootstrap(data, B=2000, seed=2027):
    """Resample whole trigger sequences and return B bootstrap accuracies."""
    rng = np.random.default_rng(seed)

    # The distinct sequence IDs, sorted so the results are reproducible.
    sequence_ids = np.sort(data["sequence_id"].unique())
    n_sequences = len(sequence_ids)

    accuracies = np.zeros(B)
    for b in range(B):
        # Draw n_sequences sequence IDs with replacement.
        drawn_ids = rng.choice(sequence_ids, size=n_sequences, replace=True)

        # Collect all the frames from each drawn sequence.
        pieces = []
        for sequence_id in drawn_ids:
            sequence_frames = data[data["sequence_id"] == sequence_id]
            pieces.append(sequence_frames)
        bootstrap_sample = pd.concat(pieces)

        # Accuracy in this bootstrap sample.
        accuracies[b] = bootstrap_sample["correct"].mean()

    return accuracies

Run it on the analysis cameras and compare with the Wilson interval. It takes about 20 seconds.

In [ ]:
k = analysis["correct"].sum()
n = len(analysis)
p_hat = k / n

wilson_lower, wilson_upper = proportion_confint(k, n, method="wilson")
accuracies = sequence_bootstrap(analysis)
boot_lower, boot_upper = np.quantile(accuracies, [0.025, 0.975])

print("Accuracy:", k, "of", n, "=", round(p_hat, 3))
print("Wilson:            ", round(wilson_lower, 3), "to", round(wilson_upper, 3))
print("Sequence bootstrap:", round(boot_lower, 3), "to", round(boot_upper, 3))

# Design effect: how much bigger the bootstrap variance is than the
# variance we would get if all frames were independent.
design_effect = np.var(accuracies, ddof=1) / (p_hat * (1 - p_hat) / n)
print("Design effect:", round(design_effect, 2))
print("Effective sample size:", round(n / design_effect, 1))

How are the errors spread across sequences?

In [ ]:
errors_per_sequence = []
for sequence_id in analysis["sequence_id"].unique():
    sequence_frames = analysis[analysis["sequence_id"] == sequence_id]
    errors_per_sequence.append(len(sequence_frames) - sequence_frames["correct"].sum())

print(pd.Series(errors_per_sequence).value_counts().sort_index())

**Your turn.** Repeat the bootstrap, the design effect, and the error table for `lab_holdout`. Why is the design effect so different?

In [ ]:
holdout_accuracies = sequence_bootstrap(lab_holdout)
# TODO: compute the Wilson interval, the bootstrap interval, the design
# effect, and the errors-per-sequence table for lab_holdout.

**What you found:** TODO

## 8. Night photos

**Why this step:** many animals at these cameras, such as coyotes, foxes, opossums, and bobcats, are most active at night, so the team cares about night accuracy. How much do we actually know about it?

In [ ]:
for time_of_day in ["day", "night"]:
    subgroup = lab_holdout[lab_holdout["day_night"] == time_of_day]
    print(time_of_day, ":", subgroup["correct"].sum(), "of", len(subgroup), "correct, in",
          subgroup["sequence_id"].nunique(), "sequences")

## 9. Stop and discuss

1. The images in a sequence are almost identical. Why did clustering still matter for the analysis cameras but not for the lab's held-out cameras?
2. Section 6 showed that most "misses" at the analysis cameras are frames where the animal had left. Is the design effect in section 7 measuring the model, the labels, or both?
3. Solving $1 + 2\rho$ = design effect for the held-out cameras gives a negative $\rho$. Is that believable, or is it noise from only 18 sequences?
4. The night frames at the held-out cameras come from only 7 sequences. How much would you trust a bootstrap interval built from 7 sequences?
5. The team wants to put the model on new cameras. What do today's intervals tell them about those cameras, and what do they not tell them?

### Discussion notes

TODO


## 10. Handoff

Write what you would put in a short report to the wildlife team about the analysis cameras:

> On [population], the estimated [quantity] was [estimate] ([interval type] 95% CI [limits]). [The main limitation, in plain words.]

**Your result:** TODO

**Homework 4 starts here.** Problem 2 reuses `sequence_bootstrap()` and the design-effect code on the homework cameras. Neither interval today says anything about cameras we have not seen.

## Exit ticket

In one sentence: what does a sequence bootstrap capture that a frame-level Wilson interval does not, and what does neither one capture?

### Exit ticket

TODO


## Before you leave

- [ ] The notebook runs from top to bottom.
- [ ] You wrote the handoff answer.
- [ ] You saved a copy. You will reuse its code in the homework or project.
